# HadithMisinfoBench — Dataset Builder & Paraphraser

Generate or expand the controlled English & Bangla Hadith misinformation benchmark (`benchmark_dataset_a.jsonl`) by cloning the project repository and running the LLM paraphraser on MAHADDAT Arabic texts with gold evidence matching.

### Workflow
1. Clone the GitHub repository into Colab.
2. Configure the API key and benchmark parameters.
3. Build the evidence store and BM25 index from the canonical Hadith corpus.
4. Run the LLM paraphrasing/generation pipeline with checkpointing.
5. Inspect benchmark statistics and sample records.
6. Package and download the generated dataset.

## Step 1 — Clone Repository & Install Dependencies

In [ ]:
# @title Clone Repository { display-mode: "form" }
import os

REPO_URL = "https://github.com/Rahatut/hadith-misinfo-bench.git" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}

repo_name = REPO_URL.split('/')[-1].replace('.git', '')

if os.path.exists(repo_name):
    print(f"Directory '{repo_name}' already exists. Pulling latest updates...")
    !cd {repo_name} && git pull
else:
    print(f"Cloning {REPO_URL} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}

os.chdir(f"/content/{repo_name}")
print(f"Current working directory: {os.getcwd()}")

print("Installing dependencies...")
!pip install -q pydantic pydantic-settings python-dotenv openai rank-bm25 tqdm pandas sentence-transformers

print("✓ Dependencies and repository ready!")

## Step 2 — Configure API Key & Benchmark Parameters

**Security:** Never hard-code a real API key into a notebook that will be shared or committed to GitHub. This cell first checks for a Colab Secret named `OPENAI_API_KEY`; if unavailable, it securely prompts for the key.

In [ ]:
# @title Benchmark Configuration { display-mode: "form" }
import os
from getpass import getpass

# ============================================================
# LLM API SETTINGS
# ============================================================

OPENAI_BASE_URL = "https://openrouter.ai/api/v1" # @param {type:"string"}

LLM_MODEL = "gpt-4o-mini" # @param ["gpt-4o-mini", "gpt-4o", "openai/gpt-4o-mini", "google/gemini-2.0-flash-001"]

# ============================================================
# DATASET GENERATION SETTINGS
# ============================================================

N_AUTHENTIC = 1000 # @param {type:"integer"}
N_FABRICATED = 1000 # @param {type:"integer"}

DATASET_SPLIT = "train" # @param ["test", "train"]
SEED = 42 # @param {type:"integer"}
CONCURRENCY = 10 # @param {type:"integer"}

OUTPUT_FILENAME = "benchmark_dataset_a.jsonl" # @param {type:"string"}

# ============================================================
# LOAD API KEY
# ============================================================

OPENAI_API_KEY = ""

# Try Google Colab Secrets first.
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY") or ""
except Exception:
    pass

# If no Colab Secret exists, securely ask for the key.
if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass("Enter your API key (input hidden): ")

if not OPENAI_API_KEY:
    raise ValueError("No API key supplied.")

# ============================================================
# ENVIRONMENT VARIABLES
# ============================================================

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

if OPENAI_BASE_URL.strip():
    os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL.strip()

os.environ["HMB_LLM_MODEL"] = LLM_MODEL

# ============================================================
# CONFIGURATION SUMMARY
# ============================================================

print("=" * 75)
print("HadithMisinfoBench Configuration")
print("=" * 75)
print(f"Target Benchmark Size : {N_AUTHENTIC} Authentic + {N_FABRICATED} Fabricated = {N_AUTHENTIC + N_FABRICATED} Total Claims")
print(f"Source Split          : MAHADDAT '{DATASET_SPLIT}' split")
print(f"LLM Paraphraser       : {LLM_MODEL}")
print(f"Concurrency            : {CONCURRENCY} workers")
print(f"Random Seed            : {SEED}")
print(f"Output Path            : data/processed/{OUTPUT_FILENAME}")
print("=" * 75)

## Step 3 — Build Evidence Store & BM25 Index

This step loads the canonical Sahih Bukhari & Muslim corpus and builds the BM25 index used for evidence retrieval and gold evidence linkage.

In [ ]:
import sys
from pathlib import Path

# Add repository source directory to Python path.
sys.path.insert(0, f"/content/{repo_name}/src")

from hadith_misinfo.config import settings
from hadith_misinfo.evidence.store import EvidenceStore
from hadith_misinfo.retrieval.bm25 import BM25Retriever

settings.ensure_dirs()

evidence_path = Path("data/processed/evidence.jsonl")
bm25_path = Path("data/indices/bm25")

print("=" * 75)
print("Evidence Store / BM25 Setup")
print("=" * 75)

if not evidence_path.exists() or not (bm25_path / "bm25.pkl").exists():

    print("Building Evidence Store from data/raw/hadith-json ...")

    store = EvidenceStore.build(
        "data/raw/hadith-json",
        verbose=True
    )

    store.save(evidence_path)

    print(f"✓ Evidence Store saved: {len(store.all_records()):,} records")

    print("\nBuilding BM25 Index ...")

    bm25 = BM25Retriever.build(
        store.all_records(),
        mode="arabic_plus_english",
        verbose=True
    )

    bm25.save(bm25_path)

    print(f"✓ BM25 Index saved to: {bm25_path}")

else:
    print("✓ Evidence store already exists:")
    print(f"  {evidence_path}")
    print("✓ BM25 index already exists:")
    print(f"  {bm25_path}")

print("=" * 75)

## Step 4 — Run Benchmark Paraphrasing & Generation

The pipeline samples authentic and fabricated claims, generates English and Bangla paraphrases, links authentic claims to gold evidence IDs, and writes output incrementally so interrupted runs can resume.

In [ ]:
# ============================================================
# VERIFY / BUILD BM25 INDEX THROUGH PROJECT CLI
# ============================================================

!python scripts/build_evidence_index.py --retriever bm25

# ============================================================
# OUTPUT FILE
# ============================================================

out_file = f"data/processed/{OUTPUT_FILENAME}"

print("\nStarting benchmark generation...")
print(f"Output: {out_file}")

# ============================================================
# BUILD BENCHMARK
# ============================================================

!python scripts/build_benchmark.py \
    --split {DATASET_SPLIT} \
    --n-authentic {N_AUTHENTIC} \
    --n-fabricated {N_FABRICATED} \
    --seed {SEED} \
    --provider openai \
    --model {LLM_MODEL} \
    --concurrency {CONCURRENCY} \
    --out {out_file}

print("\n✓ Benchmark generation command completed.")

## Step 5 — Inspect Generated Claims & Benchmark Statistics

In [ ]:
import json
import pandas as pd

# ============================================================
# LOAD JSONL
# ============================================================

records = []

with open(out_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

# ============================================================
# BASIC COUNTS
# ============================================================

auth_count = sum(
    1 for r in records
    if r.get("label") == "authentic"
)

fab_count = sum(
    1 for r in records
    if r.get("label") == "fabricated"
)

gold_count = sum(
    1 for r in records
    if r.get("gold_evidence_ids")
)

# ============================================================
# SUMMARY
# ============================================================

print("=" * 75)
print(f"BENCHMARK SUMMARY — {out_file}")
print("=" * 75)
print(f"Total Claims Generated      : {len(records):,}")
print(f"Authentic Claims            : {auth_count:,}")
print(f"Fabricated Claims           : {fab_count:,}")
print(f"Authentic/Filled Gold IDs   : {gold_count:,}")

if auth_count:
    print(f"Authentic Proportion        : {auth_count / len(records):.2%}")

if fab_count:
    print(f"Fabricated Proportion       : {fab_count / len(records):.2%}")

print("=" * 75)

# ============================================================
# DISPLAY SAMPLE CLAIMS
# ============================================================

samples = []

for r in records[:6]:
    en_claim = r.get("claims", {}).get("en", "")
    bn_claim = r.get("claims", {}).get("bn", "")

    samples.append({
        "Claim ID": r.get("claim_id", ""),
        "Label": r.get("label", ""),
        "English Claim": en_claim[:100] + ("..." if len(en_claim) > 100 else ""),
        "Bangla Claim": bn_claim[:100] + ("..." if len(bn_claim) > 100 else ""),
        "Gold IDs": "; ".join(r.get("gold_evidence_ids", []))
    })

df_samples = pd.DataFrame(samples)

display(df_samples)

## Step 6 — Additional Dataset Quality Checks

In [ ]:
# ============================================================
# QUALITY CHECKS
# ============================================================

print("=" * 75)
print("DATASET QUALITY CHECKS")
print("=" * 75)

# Missing translations
missing_en = sum(
    1 for r in records
    if not r.get("claims", {}).get("en", "").strip()
)

missing_bn = sum(
    1 for r in records
    if not r.get("claims", {}).get("bn", "").strip()
)

# Missing Arabic source
missing_ar = sum(
    1 for r in records
    if not r.get("claims", {}).get("ar", "").strip()
)

# Duplicate claim IDs
claim_ids = [r.get("claim_id") for r in records]
duplicate_ids = len(claim_ids) - len(set(claim_ids))

# Authentic claims without gold evidence
auth_without_gold = sum(
    1 for r in records
    if r.get("label") == "authentic"
    and not r.get("gold_evidence_ids")
)

print(f"Missing Arabic claims       : {missing_ar:,}")
print(f"Missing English claims      : {missing_en:,}")
print(f"Missing Bangla claims       : {missing_bn:,}")
print(f"Duplicate claim IDs         : {duplicate_ids:,}")
print(f"Authentic without Gold IDs  : {auth_without_gold:,}")

print("=" * 75)

if missing_ar == 0 and missing_en == 0 and missing_bn == 0 and duplicate_ids == 0:
    print("✓ Basic structural checks passed.")
else:
    print("⚠ Some structural issues were detected. Review the counts above.")

if auth_without_gold == 0:
    print("✓ All authentic claims have gold evidence IDs.")
else:
    print("⚠ Some authentic claims are missing gold evidence IDs.")

## Step 7 — Package & Download Generated Dataset

The export package contains the generated benchmark and the evidence store when available.

In [ ]:
import os
import shutil
from google.colab import files

# ============================================================
# CREATE EXPORT DIRECTORY
# ============================================================

export_dir = "dataset_export"
os.makedirs(export_dir, exist_ok=True)

# Remove old copies if the cell is re-run.
benchmark_export = os.path.join(export_dir, OUTPUT_FILENAME)

if os.path.exists(benchmark_export):
    os.remove(benchmark_export)

shutil.copy(out_file, benchmark_export)

# ============================================================
# COPY EVIDENCE STORE
# ============================================================

evidence_source = "data/processed/evidence.jsonl"
evidence_export = os.path.join(export_dir, "evidence.jsonl")

if os.path.exists(evidence_source):
    if os.path.exists(evidence_export):
        os.remove(evidence_export)

    shutil.copy(evidence_source, evidence_export)
    print("✓ Evidence store included in export.")
else:
    print("⚠ Evidence store not found; exporting benchmark only.")

# ============================================================
# CREATE ZIP
# ============================================================

zip_name = f"hadith_benchmark_{len(records)}_claims"

shutil.make_archive(
    zip_name,
    "zip",
    export_dir
)

zip_path = f"{zip_name}.zip"

print("=" * 75)
print(f"Export package: {zip_path}")
print("=" * 75)

files.download(zip_path)

print("\n✓ Done!")